<div dir="rtl">

# 02 — סטטיסטיקות ליגה

**פעימה A: הקבוצה בעונה ובין עונות.** השאלה: מה מבדיל אלופה מיורדת, כמה עונה אחת קשורה לבאה וכמה הכסף מסביר?

זו מחברת תיאורית: הנתונים כוללים את סוף העונה. אין כאן פיצ'רים, מודלים, כיוונון או כתיבה לדאטה הנקי. סיכום של עונה אינו קלט למודל באותה עונה; בעתיד מותר להעביר מידע מעונה קודמת דרך `prematch` בלבד.

כל החישובים מוגבלים ל-development: עונות 8–25 (2006/07–2023/24). עונות ההחזקה והעונה החיה אינן נטענות או מוצגות.

## תוכן עניינים — חמש פעימות

1. **A — הקבוצה בעונה ובין עונות** — בפרק הנוכחי.
2. B — הליגה לאורך השנים — יתווסף לאחר אישור A ומיזוג.
3. C — מהלך העונה והטבלה — טרם מומש.
4. D — המשחק והשערים — טרם מומש.
5. E — שופטים, מאמנים, אצטדיונים ומשחקים גדולים — טרם מומש.

</div>

<div dir="rtl">

### סביבת התצוגה
השאלה: באילו עונות וקבוצה נתמקד בדוגמאות? מגדירים את 2023/24 ואת מכבי חיפה, ואת אותם צבעי בית/תיקו/חוץ של מחברת 01. כל חישוב מיובא מהחבילה.

</div>

In [ ]:
import os
from pathlib import Path

import pandas as pd
import plotly.io as pio
from IPython.display import display

from ligat.analysis.league_stats import (
    persistence_summary,
    position_profile,
    promoted_value_check,
    promoted_value_summary,
    season_persistence,
    team_consistency,
    team_season_stats,
    team_style_profile,
    value_summary,
    value_vs_performance,
)
from ligat.analysis.prepare import DEVELOPMENT_SEASONS, load_analysis_tables
from ligat.data.repository import Repository
from ligat.viz.league_stats import AWAY_COLOR, DRAW_COLOR, HOME_COLOR, chapter_a_chart

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(root)
EXAMPLE_SEASON = 25
TEAM = "מכבי חיפה"
OUTCOME_COLORS = {"בית": HOME_COLOR, "תיקו": DRAW_COLOR, "חוץ": AWAY_COLOR}
pd.set_option("display.max_columns", 15)
pio.renderers.default = "plotly_mimetype+notebook"
display(
    pd.DataFrame(
        {
            "הגדרה": ["עונת דוגמה", "קבוצה", "עונות מחקר"],
            "ערך": ["2023/24", TEAM, "2006/07–2023/24 בלבד"],
        }
    )
)

<div dir="rtl">

### טעינה מוגבלת לפיתוח
השאלה: כמה משחקים ועונות באמת נכללים? קוראים דרך Repository רק את עונות הפיתוח. הטבלאות הרשמיות שנקראות בתשתית הקיימת מספקות תוויות מיקום וירידה תיאוריות בלבד; אין כאן קלט למודל או אימון. [משרת: הבנת הליגה]

</div>

In [ ]:
tables = load_analysis_tables(Repository(), seasons=DEVELOPMENT_SEASONS)
display(
    pd.DataFrame(
        {
            "תצוגה": ["משחקים", "קבוצה × משחק", "קבוצה × עונה"],
            "שורות": [len(tables.match_table), len(tables.team_matches), len(tables.team_seasons)],
        }
    )
)

<div dir="rtl">

## A — הקבוצה בעונה ובין עונות

> ### איך לקרוא את הפרק
> **נקודות מול מיקום:** נקודות למשחק מחושבות ממשחקים ששוחקו, ללא עונשים וללא חציית הנקודות של 2009/10–2010/11; המיקום הסופי רשמי ויכול לשקף גם החלטות מנהלתיות ומשחקים טכניים שהוחרגו.
>
> **n וגודל מדגם:** n הוא מספר התצפיות שהשתתפו בחישוב; מציינים בנפרד גם כמה עונות ומשחקים יש. בקבוצה עם פחות מ-30 תצפיות לא מסיקים מסקנה ולא מדרגים איכות.
>
> **רווחי 95%:** Wilson לשיעורי משחקים; לממוצעים דוגמים עונות, ולמתאמים דוגמים בלוקי עונה עם כל הקבוצות שבאותה עונה. 2,000 דגימות לממוצעים ו-1,000 לקשרים, עם זרע קבוע; הרווחים אינם מבטלים תלות בין עונות ואינם תיקון להשוואות מרובות.
>
> **חוסרים:** אין שווי קודם לעונה הראשונה ולעולות, ויש חוסר במקור. חוסר אינו אפס. סטיית תקן חסרה בפחות משלוש עונות. נתוני מחצית ותזמון שערים אינם משמשים בפרק זה, לכן חריגי המחצית אינם מסננים תוצאות משחק מלא.
>
> **שווי העולות:** ערך הכלל הוא חציון החצי התחתון לפי שווי בעונה הקודמת. העולה אינה חלק מהליגה הקודמת. השוואת אותה עונה ללא העולות היא משנית בלבד; החצי התחתון מחושב מהערכים הזמינים, ובמספר אי-זוגי נלקח floor(n/2).
>
> **קשר אינו סיבה:** שווי סוף עונה וסיכומי עונה נשארים תיאוריים. הרגרסיה מתארת את המדגם, לא בודקת חיזוי עתידי. השוואות רבות עלולות לייצר הבדלים מקריים; כל ממצא צריך בדיקה בהמשך.

### עונת דוגמה: 2023/24
**השאלה:** איך נראית עונת קבוצה במספרים?
**מה לחפש בטבלה:** השוו נקודות, הבקעה וספיגה בין הצמרת לתחתית.
**התשובה הכדורגלנית:** עונה מוצלחת משלבת צבירת נקודות עם הבקעה והגנה; מיקום לבדו אינו מתאר איך הקבוצה שיחקה. [משרת: הבנת הליגה, goals]

</div>

In [ ]:
season_stats = team_season_stats(tables)
display(
    season_stats.xs(EXAMPLE_SEASON, level="season_id")
    .sort_values("final_position")[
        [
            "team_name",
            "final_position",
            "matches_played",
            "wins",
            "draws",
            "losses",
            "points_per_match",
            "goals_scored_per_match",
            "goals_conceded_per_match",
            "home_points_per_match",
            "away_points_per_match",
            "big_win_rate",
            "btts_rate",
            "over_2_5_rate",
            "relegated",
        ]
    ]
    .round(3)
)

<div dir="rtl">

### הקבוצה שבחרנו לאורך עונות הפיתוח
**השאלה:** האם מכבי חיפה מציגה מאפיינים דומים מעונה לעונה?

**מה לחפש בגרף או בטבלה:** הסתכלו על השינויים בנקודות ועל רצפים ללא הפסד או ניצחון; אפשר לשנות TEAM בתא ההגדרות.

**התשובה הכדורגלנית:** אותה קבוצה יכולה להציג עונות שונות מאוד, ולכן נבחן בהמשך כמה להסתמך על העונה הקודמת. [prematch]

</div>

In [ ]:
display(
    season_stats.loc[
        season_stats.team_name.eq(TEAM),
        [
            "team_name",
            "final_position",
            "points_per_match",
            "goals_scored_per_match",
            "goals_conceded_per_match",
            "longest_unbeaten_run",
            "longest_winless_run",
        ],
    ]
    .tail(8)
    .round(3)
)

<div dir="rtl">

### כמה שווה כל מקום — מפרידים עידנים
**השאלה:** באיזה קצב נקודות סיימו הקבוצות בכל מקום?

**מה לחפש בגרף או בטבלה:** השוו את ממוצע הנקודות ואת הטווח בין עונות, בנפרד לכל גודל ליגה.

**התשובה הכדורגלנית:** בעידן 14 הקבוצות האלופה צברה בממוצע כ-2.28 נקודות למשחק — קצב של כ-82 נקודות ב-36 משחקים. [motivation, table_model]

</div>

In [ ]:
profiles = position_profile(tables)
display(
    profiles.xs("14_teams", level="era")[
        [
            "n",
            "points_per_match_mean",
            "points_per_match_median",
            "points_per_match_min",
            "points_per_match_max",
            "points_per_match_ci_lower",
            "points_per_match_ci_upper",
        ]
    ].round(3)
)

<div dir="rtl">

### גרף 1 — התפלגות נקודות לפי מיקום
**השאלה:** כמה רחב טווח הנקודות לכל מקום בעידן 14 הקבוצות?

**מה לחפש בגרף או בטבלה:** כל נקודה היא עונת קבוצה; קופסה גבוהה ורחבה מעידה על פיזור גדול יותר.

**התשובה הכדורגלנית:** אין מספר נקודות אחד שמבטיח אליפות או הישרדות; הסף משתנה בין העונות. [motivation, table_model]

</div>

In [ ]:
chapter_a_chart("position", season_stats).show()

<div dir="rtl">

### גרף 2 — התקפה והגנה יחד
**השאלה:** מה מבדיל בין עונת צמרת לעונת תחתית?

**מה לחפש בגרף או בטבלה:** מימין מבקיעים יותר; נמוך בגרף סופגים פחות. הצבע מציין מיקום רשמי.

**התשובה הכדורגלנית:** קבוצה מצליחה צריכה לשלב התקפה והגנה; שני הצדדים יהיו חשובים בחיזוי שערים. [goals]

</div>

In [ ]:
chapter_a_chart("attack_defense", season_stats).show()

<div dir="rtl">

### גרף 3 — בית מול חוץ
**השאלה:** איפה הקבוצות צברו יותר נקודות?

**מה לחפש בגרף או בטבלה:** מתחת לאלכסון נצברו יותר נקודות בבית, ומעליו יותר בחוץ.

**התשובה הכדורגלנית:** בממוצע עונות־קבוצה הבית הוסיף כ-0.23 נקודות למשחק; זהו קשר תיאורי, לא מדידת השפעה סיבתית. [goals, prematch]

</div>

In [ ]:
chapter_a_chart("home_away", season_stats).show()

<div dir="rtl">

### גרף 4 — ניצחונות גדולים מול צרים
**השאלה:** האם הניצחונות מגיעים בהפרש גדול או קטן?

**מה לחפש בגרף או בטבלה:** הציר האופקי מציג ניצחון בהפרש 3 ומעלה; האנכי ניצחון בהפרש שער אחד.

**התשובה הכדורגלנית:** גם ניצחון דחוק וגם ניצחון גדול מעניקים שלוש נקודות, אך הם מספרים סיפור שונה על המשחק. [goals, scores]

</div>

In [ ]:
chapter_a_chart("win_style", season_stats).show()

<div dir="rtl">

### סגנון משחק על פני לפחות חמש עונות
**השאלה:** באילו קבוצות משחקים עם שערים בשני הצדדים נפוצים יותר?

**מה לחפש בגרף או בטבלה:** השוו את שיעורי שתי הקבוצות מבקיעות ושלושה שערים ומעלה, לצד כמות העונות.

**התשובה הכדורגלנית:** קבוצות יכולות להיות פתוחות או סגורות יותר, בלי שזה כשלעצמו מעיד על איכות. [scores, goals]

</div>

In [ ]:
styles = team_style_profile(tables)
display(
    styles[
        [
            "team_name",
            "n_seasons",
            "n_matches",
            "btts_rate",
            "btts_rate_ci_lower",
            "btts_rate_ci_upper",
            "over_2_5_rate",
            "over_2_5_rate_ci_lower",
            "over_2_5_rate_ci_upper",
        ]
    ]
    .head(10)
    .round(3)
)

<div dir="rtl">

### גרף 5 — קבוצות פתוחות וסגורות
**השאלה:** כמה שונים שיעורי המשחקים הפתוחים בין הקבוצות?

**מה לחפש בגרף או בטבלה:** השוו את הנקודות ואת טווחי אי-הוודאות; טווחים חופפים מחייבים זהירות.

**התשובה הכדורגלנית:** הבדל קטן בשיעור אינו בהכרח דפוס יציב שאפשר להסתמך עליו במשחק הבא. [scores]

</div>

In [ ]:
chapter_a_chart("open_style", styles).show()

<div dir="rtl">

### רצף בין עונות — מי שיחקה גם בעונה הקודמת?
**השאלה:** האם העונה הקודמת קשורה לבאה?

**מה לחפש בגרף או בטבלה:** כל שורה מחברת אותה קבוצה בשתי עונות עוקבות, בלי לדלג מעל עונת היעדרות.

**התשובה הכדורגלנית:** לעולה אין עונה קודמת בליגת העל, ולכן צריך להבחין בינה לבין קבוצה ותיקה. [prematch, table_model]

</div>

In [ ]:
pairs = season_persistence(tables)
display(pairs.head(8).round(3))

<div dir="rtl">

### מתאם ומקדם התמדה
**השאלה:** כמה מהפער בעונה הקודמת נשמר בקשר ההיסטורי?

**מה לחפש בגרף או בטבלה:** שיפוע קרוב ל-1 מציין התמדה חזקה יותר; הסתכלו גם על המתאמים ועל טווח השיפוע.

**התשובה הכדורגלנית:** מקדם 0.77 אומר שפער של חצי נקודה למשחק קשור לכ-0.39 נקודה בעונה הבאה — לא ל-77% דיוק חיזוי. [prematch]

</div>

In [ ]:
persistence = persistence_summary(tables)
display(
    persistence[
        [
            "n",
            "n_seasons",
            "pearson",
            "spearman",
            "slope",
            "intercept",
            "slope_ci_lower",
            "slope_ci_upper",
        ]
    ].round(3)
)

<div dir="rtl">

### גרף 6 — העונה הקודמת מול הנוכחית
**השאלה:** האם הקבוצות שומרות על קצב צבירת הנקודות?

**מה לחפש בגרף או בטבלה:** השוו את הקו הכחול לאלכסון; הפיזור סביבו מראה עד כמה העונות שונות.

**התשובה הכדורגלנית:** העבר מספק כיוון, אך אינו קובע מה יקרה בעונה הבאה. [prematch]

</div>

In [ ]:
chapter_a_chart("persistence", pairs, persistence).show()

<div dir="rtl">

### כסף וביצועים — שני סוגי שווי
**השאלה:** האם שווי קודם קשור לנקודות של העונה הנוכחית?

**מה לחפש בגרף או בטבלה:** השוו בין נתח השווי בעונה הקודמת לבין נתח השווי באותה עונה; NA נשאר חסר.

**התשובה הכדורגלנית:** השווי של העונה הקודמת מתאים למחקר על חיזוי מראש; השווי העונתי הנוכחי כולל גם את המשך העונה. [goals]

</div>

In [ ]:
values = value_vs_performance(tables)
display(values.xs(EXAMPLE_SEASON, level="season_id").head(8).round(3))

<div dir="rtl">

### סיכום קשרי השווי
**השאלה:** כמה חזק הקשר בין שווי, גודל סגל ונקודות?

**מה לחפש בגרף או בטבלה:** ספירמן חיובי מציין סדר דומה בין הקבוצות; ערך שלילי מציין סדר הפוך.

**התשובה הכדורגלנית:** שווי קודם נמצא בקשר חיובי עם נקודות, וגודל הסגל בקשר שלילי חלש; קשר אינו סיבה. [goals]

</div>

In [ ]:
value_relationships = value_summary(tables)
display(
    value_relationships[
        [
            "n",
            "n_seasons",
            "pearson",
            "spearman",
            "spearman_ci_lower",
            "spearman_ci_upper",
            "slope",
            "intercept",
        ]
    ].round(3)
)

<div dir="rtl">

### גרף 7 — חריגות מול שווי קודם
**השאלה:** מי צברה יותר או פחות נקודות מהקשר הממוצע לשווי?

**מה לחפש בגרף או בטבלה:** הסימונים מצביעים על עד עשר עונות מעל הקו ועד עשר מתחתיו.

**התשובה הכדורגלנית:** חריגה יכולה לסמן עונה מפתיעה; היא אינה הוכחה לניהול טוב או רע או להצלחת חיזוי. [goals]

</div>

In [ ]:
chapter_a_chart("value", values, value_relationships).show()

<div dir="rtl">

### עולות וכלל ההשלמה
**השאלה:** האם כלל השווי נותן לעולות ערך דומה לשווי שנצפה?

**מה לחפש בגרף או בטבלה:** השוו את ערך הכלל מעונת t−1 לשווי העולה בעונה t; העמודה same_season_share_excl_promoted מציגה השוואה נפרדת לעונה t ללא העולות.

**התשובה הכדורגלנית:** הכלל מבוסס על קבוצות חלשות יחסית בליגה הקודמת; שווי העולה בעונה הנוכחית משמש רק לבדיקה בדיעבד. [prematch]

</div>

In [ ]:
promoted = promoted_value_check(tables)
display(promoted.head(10).round(4))

<div dir="rtl">

### סיכום בדיקת ההשלמה
**השאלה:** בכמה עולות ערך הכלל גבוה מהשווי שנצפה?

**מה לחפש בגרף או בטבלה:** fraction_below הוא שיעור העולות מתחת לערך הכלל; n_missing מראה כמה נשארו ללא שווי.

**התשובה הכדורגלנית:** הכלל מעריך את העולות מעט גבוה מדי במדגם הזה; זה ממצא לבדיקה, והכלל אינו משתנה כאן. [prematch]

</div>

In [ ]:
promoted_summary = promoted_value_summary(tables)
display(promoted_summary.round(4))

<div dir="rtl">

### גרף 8 — שווי העולה מול ערך הכלל
**השאלה:** כמה העולות רחוקות מהערך לפי הליגה הקודמת?

**מה לחפש בגרף או בטבלה:** מתחת לאלכסון השווי הממשי נמוך מערך הכלל; מעליו הוא גבוה ממנו.

**התשובה הכדורגלנית:** כ-72% מהעולות עם שווי זמין נמצאות מתחת לכלל — ברוב המקרים הוא מעניק להן ערך גבוה יותר מהשווי שנצפה. [prematch]

</div>

In [ ]:
chapter_a_chart("promoted", promoted).show()

<div dir="rtl">

### יציבות ויו-יו בין עונות
**השאלה:** מי מציגה קצב נקודות דומה לאורך השנים, ומי עולה שוב ושוב?

**מה לחפש בגרף או בטבלה:** הסתכלו על הממוצע, התנודתיות וכמות העונות; יו-יו מוגדרת כאן לפחות שתי עליות נצפות.

**התשובה הכדורגלנית:** למועדון עם מעט עונות יש פחות מידע על יציבות; עליות וירידות יכולות להעיד על תנודתיות לאורך זמן. [table_model, הבנת הליגה]

</div>

In [ ]:
consistency = team_consistency(tables)
display(
    consistency[
        [
            "team_name",
            "seasons",
            "mean_ppm",
            "std_ppm",
            "mean_ppm_ci_lower",
            "mean_ppm_ci_upper",
            "best_final_position",
            "worst_final_position",
            "titles",
            "top3_seasons",
            "relegations",
            "promotions",
            "is_yoyo",
        ]
    ]
    .head(10)
    .round(3)
)

<div dir="rtl">

### גרף 9 — יציבות מול ממוצע
**השאלה:** האם ממוצע גבוה מגיע עם תנודתיות קטנה?

**מה לחפש בגרף או בטבלה:** נקודה נמוכה מציינת שינוי קטן יותר בין עונות; גודל הנקודה מציין כמה עונות נצפו.

**התשובה הכדורגלנית:** יציבות וכוח ממוצע הם שני דברים שונים: קבוצה יכולה להיות יציבה גם ברמת נקודות נמוכה. [table_model]

</div>

In [ ]:
chapter_a_chart("consistency", consistency).show()

<div dir="rtl">

## מה למדנו — פעימה A

1. אלופה בעידן 14 הקבוצות צברה בממוצע כ-2.28 נקודות למשחק — קצב של כ-82 נקודות ב-36 משחקים, שנותן הקשר למאבק האליפות. [motivation, table_model]
2. הבית קשור לתוספת של כ-0.23 נקודות למשחק בממוצע עונות־קבוצה, ולכן נבחן ביצועי בית וחוץ בנפרד. [goals, prematch]
3. מקדם ההתמדה הוא כ-0.77: פער של 0.5 נקודה מעל הממוצע קשור לכ-0.39 בעונה הבאה, עם שונות רבה בין הקבוצות — זה קשר היסטורי ולא תחזית. [prematch]
4. שווי קודם נמצא בקשר חיובי עם נקודות, ספירמן כ-0.56 על 202 זוגות מלאים; שווי אותה עונה בקשר חזק יותר, כ-0.62, אך אינו מידע זמין לפני אותה עונה. [goals]
5. גודל הסגל נמצא בקשר שלילי חלש עם נקודות, ספירמן כ-‎−0.19; פירוש אפשרי שלא נבדק כאן הוא שקבוצות במשבר מחליפות יותר שחקנים ודף העונה סופר את כולם — קשר, לא סיבה. [goals, הבנת הליגה]
6. ב-26 מתוך 36 עולות עם שווי זמין, כ-72%, השווי נמוך מערך הכלל מעונת t−1, וחציון ההפרש הוא כ-‎−0.0039 מנתח הליגה; הכלל מעריך את העולות מעט גבוה מדי במדגם הזה, אבל זו בדיקת סבירות בלבד והכלל לא משתנה כאן. [prematch]

פעימות B–E טרם מומשו. נעצור לבדיקה חוזרת ולאישור מיזוג A לפני הפרק הבא.

</div>